# Sequential Neural Network

Experiment with `torch.Sequential` Neural Network model.

# Notebook Setup

## Imports

In [1]:
# Import Standard Libraries
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

# Read Data

In [2]:
# Load FashionMNIST data into a Dataset object
train_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

print('Train Data Shape: ', train_data.data.shape)
print('Test Data Shape: ', test_data.data.shape)

Train Data Shape:  torch.Size([60000, 28, 28])
Test Data Shape:  torch.Size([10000, 28, 28])


# Data Preparation

In [3]:
# Define batch size
batch_size = 64

# Define data loaders
train_dataloader = DataLoader(train_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

# Model Definition

In [4]:
# Define the device to work on
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [5]:
# Define the model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [6]:
# Instance model
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


# Model Training

## Loss & Optimiser

In [7]:
# Define loss and optimiser
loss_function = nn.CrossEntropyLoss()
optimiser = torch.optim.SGD(model.parameters(), lr=1e-3)

## Utils Functions

In [8]:
# Define a train model function
def train_model (dataloader, model, loss_function, optimiser):

    # Compute dataset size
    size = len(dataloader.dataset)

    # Switch model to training mode
    model.train()

    # Fetch the batches
    for batch, (X, y) in enumerate(dataloader):

        # Load data into device
        X, y = X.to(device), y.to(device)

        # Compute the loss
        predictions = model(X)
        loss = loss_function(predictions, y)

        # Backpropagation
        loss.backward() # Compute gradients
        optimiser.step() # Update weights
        optimiser.zero_grad() # Clear gradients buffer

        # Logging
        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [9]:
# Define a test function
def test_model(dataloader, model, loss_function):

    # Compute dataset size and number of batches
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    # Switch the model to evaluation mode
    model.eval()

    # Initialise metrics
    test_loss, correct = 0, 0

    # Feed forward input without upgrading the weights
    with torch.no_grad():

        # Compute metrics for each batch
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            predictions = model(X)
            test_loss += loss_function(predictions, y).item()
            correct += (predictions.argmax(1) == y).type(torch.float).sum().item()

    # Average of the metrics
    test_loss /= num_batches
    correct /= size

    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

## Train & Evaluate

In [11]:
# Define the epochs
epochs = 5

for epoch in range(epochs):
    print(f"Epoch {epoch+1}\n-------------------------------")
    train_model(train_dataloader, model, loss_function, optimiser)
    test_model(test_dataloader, model, loss_function)
print("Done!")

Epoch 1
-------------------------------
loss: 2.303113  [   64/60000]
loss: 2.293269  [ 6464/60000]
loss: 2.273907  [12864/60000]
loss: 2.267285  [19264/60000]
loss: 2.265402  [25664/60000]
loss: 2.229687  [32064/60000]
loss: 2.235554  [38464/60000]
loss: 2.200950  [44864/60000]
loss: 2.190758  [51264/60000]
loss: 2.170196  [57664/60000]
Test Error: 
 Accuracy: 35.9%, Avg loss: 2.164360 

Epoch 2
-------------------------------
loss: 2.171776  [   64/60000]
loss: 2.165880  [ 6464/60000]
loss: 2.107618  [12864/60000]
loss: 2.124586  [19264/60000]
loss: 2.099654  [25664/60000]
loss: 2.028145  [32064/60000]
loss: 2.059901  [38464/60000]
loss: 1.976389  [44864/60000]
loss: 1.970399  [51264/60000]
loss: 1.928555  [57664/60000]
Test Error: 
 Accuracy: 59.7%, Avg loss: 1.916949 

Epoch 3
-------------------------------
loss: 1.941329  [   64/60000]
loss: 1.921372  [ 6464/60000]
loss: 1.800791  [12864/60000]
loss: 1.845477  [19264/60000]
loss: 1.768006  [25664/60000]
loss: 1.702071  [32064/600

# Saving Model

In [12]:
# Save model weights
torch.save(model.state_dict(), "./models/model.pth")

# Model Loading

In [13]:
model_loaded = NeuralNetwork().to(device)
model_loaded.load_state_dict(torch.load("./models/model.pth", weights_only=True))

<All keys matched successfully>

In [14]:
# Inference
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model_loaded.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model_loaded(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"
